<a href="https://colab.research.google.com/github/Divyam-11/NLP_Lab/blob/Lab/NLP_LAB_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("columbine/imdb-dataset-sentiment-analysis-in-csv-format")

print("Path to dataset files:", path)
train_path = os.path.join(path, "Train.csv")
valid_path = os.path.join(path, "Valid.csv")
test_path  = os.path.join(path, "Test.csv")
# load into DataFrames
train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)
test_df  = pd.read_csv(test_path)

# quick sanity check
print(train_df.shape, valid_df.shape, test_df.shape)
train_df.head()

100%|██████████| 25.7M/25.7M [00:01<00:00, 21.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format/versions/1
(40000, 2) (5000, 2) (5000, 2)


,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:
stopwords_list = stopwords.words('english')
def preprocessing(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]','',text)
  tokenized_text = word_tokenize(text)
  temp = WordNetLemmatizer()
  preprocessed_text = [temp.lemmatize(word) for word in tokenized_text]
  return preprocessed_text

In [ ]:
train_df['text'] = train_df['text'].apply(preprocessing)
test_df['text'] = test_df['text'].apply(preprocessing)

In [ ]:
from collections import Counter
def build_vocab(texts,num_words=10000):
  counter = Counter()
  for text in texts:
    counter.update(text)
  vocab = {'PAD':0,'UNK':1}
  for word,_ in counter.most_common(num_words-2):
    vocab[word]=len(vocab)
  return vocab

In [ ]:
vocab = build_vocab(train_df['text'])

In [ ]:
vocab

{'PAD': 0,
 'UNK': 1,
 'the': 2,
 'a': 3,
 'and': 4,
 'of': 5,
 'to': 6,
 'is': 7,
 'it': 8,
 'in': 9,
 'i': 10,
 'this': 11,
 'that': 12,
 'br': 13,
 'movie': 14,
 'wa': 15,
 'film': 16,
 'for': 17,
 'with': 18,
 'but': 19,
 'on': 20,
 'not': 21,
 'you': 22,
 'are': 23,
 'he': 24,
 'his': 25,
 'have': 26,
 'one': 27,
 'be': 28,
 'at': 29,
 'all': 30,
 'by': 31,
 'an': 32,
 'who': 33,
 'they': 34,
 'from': 35,
 'like': 36,
 'so': 37,
 'there': 38,
 'just': 39,
 'or': 40,
 'her': 41,
 'about': 42,
 'if': 43,
 'ha': 44,
 'out': 45,
 'some': 46,
 'what': 47,
 'time': 48,
 'good': 49,
 'more': 50,
 'character': 51,
 'when': 52,
 'very': 53,
 'my': 54,
 'no': 55,
 'even': 56,
 'get': 57,
 'story': 58,
 'up': 59,
 'would': 60,
 'can': 61,
 'she': 62,
 'make': 63,
 'see': 64,
 'only': 65,
 'which': 66,
 'really': 67,
 'their': 68,
 'were': 69,
 'had': 70,
 'me': 71,
 'scene': 72,
 'than': 73,
 'we': 74,
 'well': 75,
 'much': 76,
 'been': 77,
 'will': 78,
 'people': 79,
 'also': 80,
 'other': 

In [ ]:
def encoding_padding(texts,vocab,maxlen=256):
  encoded = [vocab.get(token,vocab['UNK']) for token in texts]
  encoded = encoded[:maxlen]
  # Corrected: 'len' was shadowing the built-in function; changed to 'maxlen'
  padding_len = maxlen-len(encoded)
  encoded = [vocab['PAD']]*padding_len + encoded
  return torch.tensor(encoded,dtype=torch.long)

In [ ]:
class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,texts,labels,vocab):
    self.texts = texts
    self.labels = labels
    self.vocab = vocab
  def __len__(self):
    return len(self.texts)
  def __getitem__(self,index):
    encoded=encoding_padding(self.texts[index],self.vocab)
    return encoded,torch.tensor(self.labels.iloc[index],dtype=torch.float32)

In [ ]:
training_data = IMDBDataset(train_df['text'],train_df['label'],vocab)

In [ ]:
test_data = IMDBDataset(test_df['text'],test_df['label'],vocab)

In [ ]:
train_loader = DataLoader(training_data,batch_size=32,shuffle=True,num_workers=2)
test_loader = DataLoader(test_data,batch_size=32,shuffle=True,num_workers=2)

In [ ]:
len(train_loader)

1250

In [ ]:
class textclassification(nn.Module):
  def __init__(self,vocab_size,output_dim,hidden_units,num_classes):
    super(textclassification,self).__init__()
    self.embed = nn.Embedding(vocab_size,output_dim)
    self.pool = nn.AdaptiveAvgPool1d(1)
    self.hidden = nn.Linear(output_dim,hidden_units)
    self.output = nn.Linear(hidden_units,num_classes)
  def forward(self,input):
    x = self.embed(input).permute(0,2,1)
    x = self.pool(x).squeeze(2)
    x = self.hidden(x)
    x = nn.ReLU()(x)
    x = self.output(x)
    x = nn.Sigmoid()(x)
    return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
model=textclassification(len(vocab),512,256,1).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
def batch_accuracy(y_pred,y_true):
  # Threshold y_pred to get binary labels (since model uses Sigmoid output)
  predicted_labels = (y_pred > 0.5).float()
  # Compare with true labels and calculate mean accuracy
  correct_predictions = (predicted_labels == y_true).float()
  return correct_predictions.sum().item() / len(y_pred)

In [ ]:
# training loop
epochs =10
for i in range(epochs):
  loss = 0
  accuracy = 0
  model.train()
  for (x_batch,y_batch) in train_loader:
    optimizer.zero_grad()
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)
    y_pred = model(x_batch).squeeze()
    batch_loss=loss_fn(y_pred,y_batch)
    batch_loss.backward()
    optimizer.step()
    loss+=batch_loss.item()
    accuracy += batch_accuracy(y_pred,y_batch)
  print(f"Training accuracy over epoch {i+1} is {accuracy/len(train_loader):.4f} and loss is {loss/len(train_loader):.4f}")

Training accuracy over epoch 1 is 0.7014 and loss is 0.6281
Training accuracy over epoch 2 is 0.8540 and loss is 0.5701
Training accuracy over epoch 3 is 0.8803 and loss is 0.5574
Training accuracy over epoch 4 is 0.8918 and loss is 0.5527
Training accuracy over epoch 5 is 0.9013 and loss is 0.5486
Training accuracy over epoch 6 is 0.9050 and loss is 0.5468
Training accuracy over epoch 7 is 0.9140 and loss is 0.5429
Training accuracy over epoch 8 is 0.9171 and loss is 0.5412
Training accuracy over epoch 9 is 0.9150 and loss is 0.5419
Training accuracy over epoch 10 is 0.9160 and loss is 0.5413


In [ ]:
loss = 0
accuracy = 0
with torch.no_grad():
  for (x_batch,y_batch) in test_loader:
    x_batch = x_batch.to(device)
    y_batch  = y_batch.to(device)
    y_pred = model(x_batch).squeeze()
    loss+=loss_fn(y_pred,y_batch).item()
    accuracy += batch_accuracy(y_pred,y_batch)
print(f"Test accuracy is {accuracy/len(test_loader)} and the loss is {loss/len(test_loader)}")

Test accuracy is 0.8708200636942676 and loss is 0.5639816264438021
